In [1]:
import os
import sys
from pathlib import Path

# Make Spark's Python workers use THIS .venv Python (not the Microsoft Store "python3")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python     :", sys.executable)
print("JAVA_HOME  :", os.environ.get("JAVA_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

Python     : C:\Users\User\Desktop\LABS_Docs\tmdb-spark-analysis\.venv\Scripts\python.exe
JAVA_HOME  : C:\Program Files\Eclipse Adoptium\jdk-25.0.5.8-hotspot\
HADOOP_HOME: C:\hadoop


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("tmdb-env-check")   # name shown in Spark's logs
    .master("local[*]")          # run on this PC, using all CPU cores
    .getOrCreate()               # start Spark, or reuse it if already running
)
spark.sparkContext.setLogLevel("ERROR")   # hide noisy INFO/WARN messages

print("Spark version:", spark.version)

Spark version: 4.3.0


In [3]:
movies = spark.createDataFrame(
    [
        (299534, "Avengers: Endgame", 356.0, 2799.44),
        (19995,  "Avatar",            237.0, 2923.71),
    ],
    ["id", "title", "budget_musd", "revenue_musd"],
)
movies.show()

+------+-----------------+-----------+------------+
|    id|            title|budget_musd|revenue_musd|
+------+-----------------+-----------+------------+
|299534|Avengers: Endgame|      356.0|     2799.44|
| 19995|           Avatar|      237.0|     2923.71|
+------+-----------------+-----------+------------+



In [4]:
from pyspark.sql.functions import udf, col, round as spark_round
from pyspark.sql.types import DoubleType

@udf(returnType=DoubleType())
def roi_udf(revenue, budget):
    # Guard against missing values and division by zero
    if revenue is None or not budget:
        return None
    return revenue / budget

movies.withColumn(
    "roi", spark_round(roi_udf(col("revenue_musd"), col("budget_musd")), 2)
).show()

+------+-----------------+-----------+------------+-----+
|    id|            title|budget_musd|revenue_musd|  roi|
+------+-----------------+-----------+------------+-----+
|299534|Avengers: Endgame|      356.0|     2799.44| 7.86|
| 19995|           Avatar|      237.0|     2923.71|12.34|
+------+-----------------+-----------+------------+-----+



In [5]:
# Find the project root whether the notebook runs from /notebooks or the root
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "requirements.txt").exists() else Path.cwd().parent
out_path = PROJECT_ROOT / "data" / "_smoke_test"

movies.write.mode("overwrite").parquet(out_path.as_posix())
print("Rows read back:", spark.read.parquet(out_path.as_posix()).count())
print("Saved to:", out_path)

Rows read back: 2
Saved to: C:\Users\User\Desktop\LABS_Docs\tmdb-spark-analysis\data\_smoke_test


In [6]:
spark.stop()
print("Spark stopped cleanly ✅")

Spark stopped cleanly ✅
